# Module 3 — Support Assistant

Run every cell. This notebook creates the required `support_assistant/` folder and downloads it as a ZIP. The graded default is `MOCK_LLM=1`: no API key and no LLM-provider call.

In [1]:
!pip -q install fastapi 'uvicorn[standard]' langgraph chromadb sentence-transformers 'pydantic>=2' httpx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [2]:
from pathlib import Path
ROOT = Path('support_assistant'); DOCS = ROOT/'docs'; DOCS.mkdir(parents=True, exist_ok=True)
documents = {
'doc_01.txt': "Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.",
'doc_02.txt': "Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto.",
'doc_03.txt': "Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period.",
'doc_04.txt': "Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue.",
'doc_05.txt': "Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee.",
'doc_06.txt': "If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items without requiring the customer to return the original item, unless the order value exceeds INR 1000, in which case a photo of the issue must be submitted through the report form before a replacement or refund is processed.",
'doc_07.txt': "Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law.",
'doc_08.txt': "Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered."}
for name, text in documents.items(): (DOCS/name).write_text(text, encoding='utf-8')
print('Created', len(documents), 'exact corpus files.')

Created 8 exact corpus files.


## Structured prompt template

This contains role, context, task, format, length, an explicit negative constraint, and a few-shot example. It is used by the optional real-LLM extension only; mock mode remains the required baseline.

In [3]:
PROMPT_TEMPLATE = '''Role: You are Zepto's policy support assistant.
Context: {context}
Task: Answer the customer's question using only the provided context.
Format: Return concise plain text with the relevant policy details.
Length: Maximum 80 words.
Constraint: Do not answer using information not present in the provided context.
Few-shot example:
Question: What is the standard delivery fee?
Context: Standard delivery is free above INR 149; otherwise it costs INR 25.
Answer: Standard delivery is free on orders over INR 149 and costs INR 25 below that threshold.
Question: {question}
Answer:'''
print(PROMPT_TEMPLATE)

Role: You are Zepto's policy support assistant.
Context: {context}
Task: Answer the customer's question using only the provided context.
Format: Return concise plain text with the relevant policy details.
Length: Maximum 80 words.
Constraint: Do not answer using information not present in the provided context.
Few-shot example:
Question: What is the standard delivery fee?
Context: Standard delivery is free above INR 149; otherwise it costs INR 25.
Answer: Standard delivery is free on orders over INR 149 and costs INR 25 below that threshold.
Question: {question}
Answer:


In [4]:
%%writefile support_assistant/main.py
from __future__ import annotations
import os
from pathlib import Path
from typing import Literal, TypedDict
import chromadb
from fastapi import FastAPI
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer
from langgraph.graph import END, StateGraph
ROOT=Path(__file__).parent; DOCS_DIR=ROOT/'docs'; CHROMA_DIR=ROOT/'chroma_db'; COLLECTION='zepto_policy_chunks'
MOCK_LLM=os.getenv('MOCK_LLM','1') != '0'
KEYWORDS=('delivery','return','refund','membership','tracking','cancel','gift card','support hours')
class AskRequest(BaseModel): query:str=Field(min_length=1,max_length=1000)
class AskResponse(BaseModel): answer:str; sources:list[str]; confidence:float=Field(ge=0,le=1)
class State(TypedDict,total=False): query:str; intent:Literal['policy_question','general_question']; response:AskResponse
embedder=None; coll=None
def get_collection():
 global embedder,coll
 if coll is not None: return coll
 embedder=SentenceTransformer('all-MiniLM-L6-v2')
 client=chromadb.PersistentClient(path=str(CHROMA_DIR))
 try: client.delete_collection(COLLECTION)
 except Exception: pass
 coll=client.create_collection(COLLECTION,metadata={'hnsw:space':'cosine'})
 paths=sorted(DOCS_DIR.glob('doc_*.txt'))
 if len(paths)!=8: raise RuntimeError('Expected 8 corpus files')
 texts=[p.read_text(encoding='utf-8') for p in paths]; ids=[p.stem for p in paths]
 coll.add(ids=ids,documents=texts,metadatas=[{'document_id':i} for i in ids],embeddings=embedder.encode(texts,normalize_embeddings=True).tolist())
 return coll
def classify_intent(state):
 q=state['query'].lower(); return {'intent':'policy_question' if any(k in q for k in KEYWORDS) else 'general_question'}
def retrieve_and_answer(state):
 c=get_collection(); r=c.query(query_embeddings=embedder.encode([state['query']],normalize_embeddings=True).tolist(),n_results=3,include=['documents','metadatas'])
 ids=[m['document_id'] for m in r['metadatas'][0]]; top=r['documents'][0][0]
 if MOCK_LLM: response=AskResponse(answer=f'Based on the retrieved context: {top[:200]}',sources=ids,confidence=1.0)
 else: raise RuntimeError('Optional MOCK_LLM=0 real LLM extension is not configured.')
 return {'response':response}
def direct_answer(state):
 if MOCK_LLM: response=AskResponse(answer='I can only answer questions about Zepto policies right now.',sources=[],confidence=1.0)
 else: raise RuntimeError('Optional MOCK_LLM=0 real LLM extension is not configured.')
 return {'response':response}
def route(state): return 'retrieve_and_answer' if state['intent']=='policy_question' else 'direct_answer'
g=StateGraph(State); g.add_node('classify_intent',classify_intent); g.add_node('retrieve_and_answer',retrieve_and_answer); g.add_node('direct_answer',direct_answer); g.set_entry_point('classify_intent'); g.add_conditional_edges('classify_intent',route,{'retrieve_and_answer':'retrieve_and_answer','direct_answer':'direct_answer'}); g.add_edge('retrieve_and_answer',END); g.add_edge('direct_answer',END); graph=g.compile()
app=FastAPI(title='Zepto Policy Support Assistant')
@app.post('/ask',response_model=AskResponse)
def ask(request:AskRequest): return graph.invoke({'query':request.query})['response']


Writing support_assistant/main.py


In [5]:
# Demonstrate both required routes with MOCK_LLM left at its default. First run downloads the open-source embedding model.
import sys; sys.path.insert(0, 'support_assistant')
from fastapi.testclient import TestClient
from main import app
client = TestClient(app)
policy = client.post('/ask', json={'query':'What is the delivery fee?'})
general = client.post('/ask', json={'query':'What is the capital of India?'})
print('Policy query:', policy.status_code, policy.json())
print('General query:', general.status_code, general.json())
assert policy.status_code == 200 and 'doc_01' in policy.json()['sources']
assert general.status_code == 200 and general.json()['sources'] == []

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Policy query: 200 {'answer': "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del", 'sources': ['doc_01', 'doc_05', 'doc_02'], 'confidence': 1.0}
General query: 200 {'answer': 'I can only answer questions about Zepto policies right now.', 'sources': [], 'confidence': 1.0}


In [6]:
# Create Docker/requirements/README, then download the entire submission folder.
(ROOT/'requirements.txt').write_text('fastapi>=0.110\nuvicorn[standard]>=0.27\nlanggraph>=0.2\npydantic>=2.6\nchromadb>=0.5\nsentence-transformers>=3.0\ntorch>=2.2\n')
(ROOT/'Dockerfile').write_text('FROM python:3.11-slim\nWORKDIR /app\nCOPY requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\nCOPY . .\nEXPOSE 7860\nCMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"]\n')
readme = '''# Zepto Policy Support Assistant

## Architecture
Ingestion: `docs/doc_01.txt`–`doc_08.txt` are read as one chunk each. Embedding: `get_collection()` embeds them locally with `all-MiniLM-L6-v2`. Retrieval: ChromaDB collection `zepto_policy_chunks` returns the three nearest chunks in `retrieve_and_answer`. Generation: the same LangGraph node returns the mock `Based on the retrieved context: ...` answer. `classify_intent` conditionally routes policy queries to retrieval and general questions to `direct_answer`.

`MOCK_LLM=1` is the default graded path: no LLM API calls are made. In both answer nodes the generation step branches on `MOCK_LLM`; `MOCK_LLM=0` is reserved for an optional real-LLM extension. The Pydantic response always contains `answer`, `sources`, and `confidence`.

## Run
`pip install -r requirements.txt` then `uvicorn main:app --host 0.0.0.0 --port 7860`.

## Docker
`docker build -t zepto-support-assistant .` then `docker run --rm -p 7860:7860 zepto-support-assistant`.

Example mock responses are printed in the executed notebook for a policy query and an unrelated general query.
'''
(ROOT/'README.md').write_text(readme)
import shutil
shutil.make_archive('support_assistant_submission', 'zip', '.', 'support_assistant')
from google.colab import files
files.download('support_assistant_submission.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>